# 03 · Are the per-fiscal-year contract files needed?

**Stage 2, checklist question 7.** CanadaBuys publishes the contract history twice: as two "complete" files
(legacy and current) and as one file per fiscal year. If the yearly files held rows the complete files lack,
loading only the complete files would silently drop contracts. So they are compared **row by row** on
(contract number, amendment number), the key notebook 01 showed to be unique.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from spendpipe import sources as S, eda
eda.apply_style()
pd.set_option("display.max_columns", 30, "display.width", 160, "display.max_colwidth", 60)

In [2]:
complete = pd.concat([S.load("contracts_legacy"), S.load("contracts_current")], ignore_index=True)
yearly = S.load_yearly()
headers_match = {p.name: pd.read_csv(p, nrows=0, encoding="utf-8-sig").columns.tolist() ==
                 pd.read_csv(S.find_file("contracts_current"), nrows=0, encoding="utf-8-sig").columns.tolist()
                 for p in sorted((S.RAW_DIR / "yearly").glob("*.csv"))}
print(f"yearly files: {len(headers_match)} | all have the same header as the complete files: {all(headers_match.values())}")
print(f"rows: yearly {len(yearly):,} | complete {len(complete):,}")

yearly files: 19 | all have the same header as the complete files: True
rows: yearly 509,496 | complete 509,389


How is each yearly file cut? By award date, amendment date, or publication date?

In [3]:
t = S.parse_types(yearly)
t.groupby("source").agg(rows=("contract_number", "size"), award_first=("award_date", "min"),
                        award_last=("award_date", "max"), no_award=("award_date", lambda s: s.isna().sum()))

,rows,award_first,award_last,no_award
source,,,,
2009-2010-contractHistory-contratsOctroyes.csv,69168,2009-04-01,2010-03-31,15543
2009-jan-Mar-contractHistory-contratsOctroyes.csv,24143,2009-01-01,2009-03-31,6843
2010-2011-contractHistory-contratsOctroyes.csv,59331,2010-04-01,2011-03-31,10104
2011-2012-contractHistory-contratsOctroyes.csv,52394,2011-04-01,2012-03-31,6933
2012-2013-contractHistory-contratsOctroyes.csv,39304,2012-04-01,2013-03-31,5999
2013-2014-contractHistory-contratsOctroyes.csv,32484,2013-04-01,2014-03-31,4062
2014-2015-contractHistory-contratsOctroyes.csv,28148,2014-04-01,2015-03-31,3305
2015-2016-contractHistory-contratsOctroyes.csv,31829,2015-04-01,2016-03-31,3250
2016-2017-contractHistory-contratsOctroyes.csv,28105,2016-04-01,2017-03-31,2883


In [4]:
key = ["contract_number", "amendment_number"]
print(f"repeated (contract, amendment) pairs within the yearly files: {yearly.duplicated(key).sum():,}")
m = yearly.merge(complete, on=key, how="outer", suffixes=("_y", "_c"), indicator=True)
m["_merge"].value_counts().rename({"left_only": "only in yearly files", "right_only": "only in complete files",
                                   "both": "in both"})

repeated (contract, amendment) pairs within the yearly files: 0


_merge
in both                   509388
only in yearly files         108
only in complete files         1
Name: count, dtype: int64

In [5]:
only = m[m._merge != "both"]
print("rows in one set only, by award fiscal year file / contract-number prefix:")
print(only.assign(file=only.source_y.fillna(only.source_c), prefix=only.contract_number.str[:2])
      .groupby(["_merge", "file", "prefix"], observed=True).size())
only[["contract_number", "amendment_number", "award_date_y", "amendment_date_y", "award_date_c", "amendment_date_c"]].head(6)

rows in one set only, by award fiscal year file / contract-number prefix:
_merge      file                                            prefix
left_only   2025-2026-contractHistory-contratsOctroyes.csv  CW        108
right_only  contracts_current                               CW          1
dtype: int64


,contract_number,amendment_number,award_date_y,amendment_date_y,award_date_c,amendment_date_c
95209,CW2323667,001,2025-05-28,2025-05-20,NaN,NaN
95999,CW2329445,002,2025-10-01,2026-10-01,NaN,NaN
100358,CW2351260,002,NaN,NaN,2025-05-29,2025-05-30
102159,CW2363740,006,2025-08-12,2026-10-19,NaN,NaN
103399,CW2370332,001,2025-09-04,2025-09-04,NaN,NaN
103843,CW2373006,000,2025-05-28,NaN,NaN,NaN


For rows present in both, does any field differ?

In [6]:
both = m[m._merge == "both"]
fields = [c for c in S.CONTRACT_COLUMNS if c not in key]
diff = {f: int(((both[f + "_y"] != both[f + "_c"]) & ~(both[f + "_y"].isna() & both[f + "_c"].isna())).sum())
        for f in fields}
pd.Series({f: n for f, n in diff.items() if n}, name="matched rows that differ").to_frame()

,matched rows that differ
title,4
number_of_records,214
publication_date,15
award_date,1
amendment_date,1
start_date,1
total_value,125
procurement_category,2
selection_criteria,9
client_department,241


In [7]:
differs = pd.concat([both[f + "_y"].ne(both[f + "_c"]) & ~(both[f + "_y"].isna() & both[f + "_c"].isna())
                     for f in fields], axis=1).any(axis=1)
d = both[differs]
print(f"matched rows with any difference: {len(d):,} ({len(d) / len(both):.3%})")
print("by yearly file:", d.source_y.value_counts().to_dict())
cd = d[d.client_department_y != d.client_department_c]
cd[["contract_number", "client_department_y", "client_department_c"]].head(4)

matched rows with any difference: 468 (0.092%)
by yearly file: {'2025-2026-contractHistory-contratsOctroyes.csv': 286, '2023-2024-contractHistory-contratsOctroyes.csv': 66, '2024-2025-contractHistory-contratsOctroyes.csv': 63, '2026-2027-contractHistory-contratsOctroyes.csv': 20, '2022-2023-contractHistory-contratsOctroyes.csv': 16, '2020-2021-contractHistory-contratsOctroyes.csv': 8, '2021-2022-contractHistory-contratsOctroyes.csv': 4, '2013-2014-contractHistory-contratsOctroyes.csv': 3, '2019-2020-contractHistory-contratsOctroyes.csv': 2}


,contract_number,client_department_y,client_department_c
74777,87055-210108/001/PI,Department of Public Works and Government Services (PSPC...,Canadian Nuclear Safety Commission (CNSC) / Department o...
87161,CW2225359,Impact Assessment Agency of Canada (IAAC) / Department o...,Department of Public Works and Government Services (PSPC...
87331,CW2227558,Department of Public Works and Government Services (PSPC...,Impact Assessment Agency of Canada (IAAC) / Department o...
87411,CW2228027,Impact Assessment Agency of Canada (IAAC) / Department o...,Department of Public Works and Government Services (PSPC...


In [8]:
changed = both[both.total_value_y != both.total_value_c]
print(f"contracts whose total value differs: {changed.contract_number.nunique()} | award years: "
      f"{changed.award_date_y.str[:7].value_counts().to_dict()}")
newest = pd.to_datetime(yearly.amendment_date).max(), pd.to_datetime(complete.amendment_date).max()
print("latest amendment date, ignoring future-dated values (> today):",
      pd.to_datetime(yearly.amendment_date)[lambda s: s <= pd.Timestamp.today()].max().date(), "(yearly) vs",
      pd.to_datetime(complete.amendment_date)[lambda s: s <= pd.Timestamp.today()].max().date(), "(complete)")

contracts whose total value differs: 49 | award years: {'2025-11': 34, '2025-08': 18, '2026-01': 18, '2025-10': 14, '2025-12': 10, '2025-04': 8, '2025-09': 8, '2026-03': 6, '2026-02': 5, '2025-07': 2, '2025-06': 2}


latest amendment date, ignoring future-dated values (> today): 2026-10-20 (yearly) vs 2026-10-06 (complete)


## Findings

1. **Same rows.** The 19 yearly files (cut by award fiscal year) and the two complete files hold the same records:
   509,388 (contract, amendment) rows match. 108 rows are only in the yearly files and 1 only in the complete files,
   all `CW...` contracts in FY2025-26.
2. **Small differences, from a later refresh.** 468 matched rows (0.09%) differ in some field, mostly in the last
   three fiscal years. The yearly files contain amendments up to 2026-10-20, the complete files up to 2026-10-06,
   so the yearly files were refreshed more recently. 49 recent contracts have a different total value for that reason.
3. **Multi-department lists have no fixed order.** Most client-department differences are the same departments
   listed in a different order ("A / B" vs "B / A"). Any rule that groups by client department must split and sort
   these lists first (stage 4).
4. **Conclusion for the decision:** the complete files lose nothing except about two weeks of the newest
   amendments, in months that are already provisional because of posting lag. Using them alone is safe.